# Lakehouse Agent - Prerequisites Setup

This notebook helps you set up the initial configuration in AWS Systems Manager (SSM) Parameter Store.

**What this notebook does:**
- Validates your AWS credentials and region
- Auto-detects AWS Account ID
- Creates initial SSM parameters with the `lh_` prefix
- Validates the configuration

**Prerequisites:**
- AWS credentials configured (via AWS CLI or environment variables)
- Python 3.10 or later
- boto3 installed: `pip install boto3`

**IAM Permissions Required:**
- `ssm:PutParameter`
- `ssm:GetParameter`
- `sts:GetCallerIdentity`

In [1]:
import boto3
import json
from datetime import datetime

print("✅ Imports successful")

✅ Imports successful


## Step 1: Validate AWS Credentials

First, let's verify your AWS credentials are configured correctly.

In [2]:
# Initialize AWS clients
try:
    sts_client = boto3.client('sts')
    ssm_client = boto3.client('ssm')
    
    # Get caller identity
    identity = sts_client.get_caller_identity()
    
    AWS_ACCOUNT_ID = identity['Account']
    AWS_REGION = boto3.Session().region_name or 'us-east-1'
    
    print("✅ AWS Credentials Validated")
    print(f"   Account ID: {AWS_ACCOUNT_ID}")
    print(f"   Region: {AWS_REGION}")
    print(f"   User/Role: {identity['Arn']}")
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("\nPlease configure AWS credentials:")
    print("  aws configure")
    print("  or set environment variables: AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY")

✅ AWS Credentials Validated
   Account ID: XXXXXXXXXXXX
   Region: us-east-1
   User/Role: arn:aws:sts::XXXXXXXXXXXX:assumed-role/AWSReservedSSO_AdministratorAccess_69906e41ccdeafff/skoppar+agentcore


## Step 2: Define Initial Configuration

Set your initial configuration values. These will be stored in SSM Parameter Store.

**Note:** AWS_REGION and AWS_ACCOUNT_ID are auto-detected and NOT stored in SSM.

In [3]:
# Initial configuration - UPDATE THESE VALUES
config = {
    # S3 Configuration
    'S3_BUCKET_NAME': 'lk-agent',  # CHANGE THIS
    'S3_CLAIMS_PREFIX': 'lakehouse-data/claims/',
    'S3_USERS_PREFIX': 'lakehouse-data/users/',
    'S3_ATHENA_RESULTS_PREFIX': 'athena-results/',
    
    # Athena Configuration
    'ATHENA_DATABASE_NAME': 'lakehouse_db',
    'ATHENA_WORKGROUP': 'primary',
    
    # Security Configuration
    'SECURITY_MODE': 'lakeformation',
    'LOCAL_DEVELOPMENT': 'false',
    'LOG_LEVEL': 'INFO',
    
    # Test Users
    'TEST_USER_1': 'user001@example.com',
    'TEST_USER_2': 'user002@example.com',
    'TEST_USER_3': 'adjuster001@example.com',
    'TEST_PASSWORD': 'TempPass123!'
}

print("📋 Initial Configuration:")
for key, value in config.items():
    print(f"   {key}: {value}")

📋 Initial Configuration:
   S3_BUCKET_NAME: lk-agent
   S3_CLAIMS_PREFIX: lakehouse-data/claims/
   S3_USERS_PREFIX: lakehouse-data/users/
   S3_ATHENA_RESULTS_PREFIX: athena-results/
   ATHENA_DATABASE_NAME: lakehouse_db
   ATHENA_WORKGROUP: primary
   SECURITY_MODE: lakeformation
   LOCAL_DEVELOPMENT: false
   LOG_LEVEL: INFO
   TEST_USER_1: user001@example.com
   TEST_USER_2: user002@example.com
   TEST_USER_3: adjuster001@example.com
   TEST_PASSWORD: TempPass123!


## Step 3: Create SSM Parameters

This will create all parameters in SSM Parameter Store with the `lh_` prefix.

**Sensitive parameters** (containing SECRET, PASSWORD, KEY) will be created as SecureString.

In [4]:
def is_sensitive(key):
    """Check if parameter should be SecureString"""
    sensitive_keywords = ['SECRET', 'PASSWORD', 'KEY', 'TOKEN']
    return any(keyword in key.upper() for keyword in sensitive_keywords)

def create_ssm_parameter(key, value, overwrite=False):
    """Create or update SSM parameter"""
    # Convert to SSM parameter name (lowercase with lh_ prefix)
    param_name = f"lh_{key.lower()}"
    param_type = 'SecureString' if is_sensitive(key) else 'String'
    
    try:
        ssm_client.put_parameter(
            Name=param_name,
            Value=str(value),
            Type=param_type,
            Description=f"Lakehouse Agent - {key}",
            Overwrite=overwrite
        )
        return True, param_type
    except ssm_client.exceptions.ParameterAlreadyExists:
        return False, param_type
    except Exception as e:
        print(f"❌ Error creating {param_name}: {e}")
        return None, param_type

# Create parameters
print("🔄 Creating SSM Parameters...\n")
created = 0
skipped = 0
failed = 0

for key, value in config.items():
    result, param_type = create_ssm_parameter(key, value, overwrite=False)
    param_name = f"lh_{key.lower()}"
    
    if result is True:
        print(f"✅ Created {param_name} ({param_type})")
        created += 1
    elif result is False:
        print(f"⏭️  Skipped {param_name} (already exists)")
        skipped += 1
    else:
        failed += 1

print(f"\n📊 Summary:")
print(f"   Created: {created}")
print(f"   Skipped: {skipped}")
print(f"   Failed: {failed}")

🔄 Creating SSM Parameters...

✅ Created lh_s3_bucket_name (String)
✅ Created lh_s3_claims_prefix (String)
✅ Created lh_s3_users_prefix (String)
✅ Created lh_s3_athena_results_prefix (String)
✅ Created lh_athena_database_name (String)
✅ Created lh_athena_workgroup (String)
✅ Created lh_security_mode (String)
✅ Created lh_local_development (String)
✅ Created lh_log_level (String)
✅ Created lh_test_user_1 (String)
✅ Created lh_test_user_2 (String)
✅ Created lh_test_user_3 (String)
✅ Created lh_test_password (SecureString)

📊 Summary:
   Created: 13
   Skipped: 0
   Failed: 0


## Step 4: Validate SSM Parameters

Let's verify all parameters were created successfully.

In [5]:
def validate_ssm_parameters():
    """Validate all required parameters exist in SSM"""
    print("🔍 Validating SSM Parameters...\n")
    
    missing = []
    found = []
    
    for key in config.keys():
        param_name = f"lh_{key.lower()}"
        try:
            response = ssm_client.get_parameter(Name=param_name)
            param_type = response['Parameter']['Type']
            
            if param_type == 'SecureString':
                value = '****** (encrypted)'
            else:
                value = response['Parameter']['Value']
            
            print(f"✅ {param_name}: {value}")
            found.append(param_name)
        except ssm_client.exceptions.ParameterNotFound:
            print(f"❌ {param_name}: NOT FOUND")
            missing.append(param_name)
    
    print(f"\n📊 Validation Summary:")
    print(f"   Found: {len(found)}")
    print(f"   Missing: {len(missing)}")
    
    if missing:
        print(f"\n⚠️  Missing parameters: {', '.join(missing)}")
        return False
    else:
        print(f"\n✅ All parameters validated successfully!")
        return True

validate_ssm_parameters()

🔍 Validating SSM Parameters...

✅ lh_s3_bucket_name: lk-agent
✅ lh_s3_claims_prefix: lakehouse-data/claims/
✅ lh_s3_users_prefix: lakehouse-data/users/
✅ lh_s3_athena_results_prefix: athena-results/
✅ lh_athena_database_name: lakehouse_db
✅ lh_athena_workgroup: primary
✅ lh_security_mode: lakeformation
✅ lh_local_development: false
✅ lh_log_level: INFO
✅ lh_test_user_1: user001@example.com
✅ lh_test_user_2: user002@example.com
✅ lh_test_user_3: adjuster001@example.com
✅ lh_test_password: ****** (encrypted)

📊 Validation Summary:
   Found: 13
   Missing: 0

✅ All parameters validated successfully!


True

## Step 5: Test Configuration Loading

Test that the configuration can be loaded using the config module.

In [6]:
# Test loading configuration
try:
    from config import config as app_config
    
    print("✅ Configuration loaded successfully\n")
    print("📋 Configuration Status:")
    app_config.print_status()
    
except Exception as e:
    print(f"❌ Error loading configuration: {e}")
    print("\nMake sure you're running this notebook from the lakehouse-agent directory")

✅ Configuration loaded successfully

📋 Configuration Status:

Configuration Status

📍 Configuration Source: SSM Parameter Store
   Region: us-east-1
   Account ID: XXXXXXXXXXXX
   Parameter Prefix: lh_

✅ = Configured | ❌ = Missing

✅ AWS_REGION: us-east-1
✅ S3_BUCKET_NAME: lk-agent
✅ ATHENA_DATABASE_NAME: lakehouse_db
❌ COGNITO_USER_POOL_ID: 
❌ COGNITO_APP_CLIENT_ID: 
❌ COGNITO_DOMAIN: 
❌ RLS_ROLE_ARN: 

Security Mode: lakeformation
Overall Status: ❌ Invalid - Check missing fields



## Step 6: Update Parameters (Optional)

If you need to update existing parameters, use this cell.

In [ ]:
# Update specific parameters
updates = {
    # Uncomment and modify as needed
    # 'S3_BUCKET_NAME': 'new-bucket-name',
    # 'LOG_LEVEL': 'DEBUG',
}

if updates:
    print("🔄 Updating SSM Parameters...\n")
    for key, value in updates.items():
        result, param_type = create_ssm_parameter(key, value, overwrite=True)
        param_name = f"lh_{key.lower()}"
        if result is not None:
            print(f"✅ Updated {param_name} = {value}")
else:
    print("ℹ️  No updates specified")

## Step 7: Export Configuration (Backup)

Export current SSM parameters to a file for backup.

In [ ]:
def export_ssm_parameters(output_file='ssm_backup.json', include_secrets=False):
    """Export SSM parameters to JSON file"""
    print(f"📤 Exporting SSM parameters to {output_file}...\n")
    
    export_data = {
        'exported_at': datetime.now().isoformat(),
        'aws_account_id': AWS_ACCOUNT_ID,
        'aws_region': AWS_REGION,
        'parameters': {}
    }
    
    for key in config.keys():
        param_name = f"lh_{key.lower()}"
        try:
            response = ssm_client.get_parameter(
                Name=param_name,
                WithDecryption=include_secrets
            )
            param = response['Parameter']
            
            if param['Type'] == 'SecureString' and not include_secrets:
                value = '****** (encrypted)'
            else:
                value = param['Value']
            
            export_data['parameters'][param_name] = {
                'value': value,
                'type': param['Type']
            }
            print(f"✅ Exported {param_name}")
        except Exception as e:
            print(f"⚠️  Skipped {param_name}: {e}")
    
    # Write to file
    with open(output_file, 'w') as f:
        json.dump(export_data, f, indent=2)
    
    print(f"\n✅ Exported {len(export_data['parameters'])} parameters to {output_file}")

# Export (without decrypting secrets)
export_ssm_parameters('ssm_backup.json', include_secrets=False)

## Step 8: List All Lakehouse Parameters

View all parameters with the `lh_` prefix.

In [ ]:
def list_lakehouse_parameters():
    """List all parameters with lh_ prefix"""
    print("📋 All Lakehouse Parameters:\n")
    
    try:
        # Get all parameters with lh_ prefix
        paginator = ssm_client.get_paginator('describe_parameters')
        page_iterator = paginator.paginate(
            ParameterFilters=[
                {
                    'Key': 'Name',
                    'Option': 'BeginsWith',
                    'Values': ['lh_']
                }
            ]
        )
        
        count = 0
        for page in page_iterator:
            for param in page['Parameters']:
                print(f"  • {param['Name']} ({param['Type']})")
                count += 1
        
        print(f"\n📊 Total: {count} parameters")
        
    except Exception as e:
        print(f"❌ Error: {e}")

list_lakehouse_parameters()

## Next Steps

✅ **Prerequisites Complete!**

Your SSM Parameter Store is now configured. You can proceed with the deployment:

1. **Deploy Athena Database**
   ```bash
   cd athena-setup
   python setup_athena.py
   ```

2. **Deploy Lake Formation RLS**
   ```bash
   python setup_lake_formation.py
   ```
   
   Then save the RLS_ROLE_ARN to SSM:
   ```python
   ssm_client.put_parameter(
       Name='lh_rls_role_arn',
       Value='arn:aws:iam::ACCOUNT:role/lakehouse-rls-role',
       Type='String',
       Overwrite=True
   )
   ```

3. **Continue with remaining deployment steps** as documented in README.md

### Useful Commands

**Validate configuration:**
```bash
python ssm_migrate.py validate
```

**View configuration status:**
```python
from config import config
config.print_status()
```